# CSV Preprocessing

## Articles from Journals

Inspecting the first rows and most importantly, headers

In [ ]:
import pandas as pd

# The tool generates separate header and data files. 
# Adjust these names if you want to check proceedings or inproceedings instead.
header_file = 'output_dblp/output_article_header.csv'
data_file = 'output_dblp/output_article.csv'

# 1. Extract the Neo4j-formatted headers
with open(header_file, 'r', encoding='utf-8') as f:
    headers = f.read().strip().split(',')

# 2. Load just the first 5 rows of the actual data using those headers
df_article = pd.read_csv(data_file, names=headers, nrows=5, on_bad_lines='skip')

# 3. Preview the structure
print("\n--- NEO4J COLUMNS ---")
for col in headers:
    print(col)

print("\n--- FIRST 5 ROWS ---")
print(df_article.head())



--- NEO4J COLUMNS ---
article:ID;author:string[];author-aux:string;author-orcid:string[];booktitle:string;cdate:date;cdrom:string;cite:string[];cite-label:string[];crossref:string;editor:string[];editor-orcid:string[];ee:string[];ee-type:string[];i:string[];journal:string;key:string;mdate:date;month:string;note:string[];note-label:string;note-type:string[];number:string;pages:string;publisher:string;publnr:string;publtype:string;stream:string;sub:string[];sup:string[];title:string;title-bibtex:string;tt:string[];url:string[];volume:string;year:int

--- FIRST 5 ROWS ---
  article:ID;author:string[];author-aux:string;author-orcid:string[];booktitle:string;cdate:date;cdrom:string;cite:string[];cite-label:string[];crossref:string;editor:string[];editor-orcid:string[];ee:string[];ee-type:string[];i:string[];journal:string;key:string;mdate:date;month:string;note:string[];note-label:string;note-type:string[];number:string;pages:string;publisher:string;publnr:string;publtype:string;stream:str

5

Removing error / notes included within the first rows and generating a clean file

In [5]:
# 1. Get headers
with open(header_file, 'r', encoding='utf-8') as f:
    headers = f.read().strip().split(';') # Note: Your output showed ';' as the separator

# 2. Read a chunk of the data (e.g., first 100,000 rows to find good data)
df = pd.read_csv(data_file, sep=';', names=headers, nrows=100000, on_bad_lines='skip', low_memory=False)

# 3. Filter out the "junk" records
# Keep rows where the 'key' column does NOT contain 'dblpnote'
df_clean = df[~df['key:string'].astype(str).str.contains('dblpnote', na=False)]

# Keep rows that actually have a title and a year
df_clean = df_clean.dropna(subset=['title:string', 'year:int'])

# Print the total number of clean articles before subsetting
print(f"Total number of clean articles: {len(df_clean)}")

# 4. Take a final sample of 10,000 clean papers for your lab
df_final = df_clean.head(10000)

# 5. Save the clean data to a new CSV
# df_final.to_csv('clean_articles.csv', index=False, sep=';')
print(f"Successfully saved {len(df_final)} clean papers!")

Total number of clean articles: 99954
Successfully saved 10000 clean papers!


Extraction of journals and volumes from the article information

In [ ]:
import pandas as pd

# Load the clean articles we just made
df = pd.read_csv('clean_articles.csv', sep=';')

# --- 1. EXTRACT JOURNALS ---
# Drop duplicates to get a unique list of journals
journals = df[['journal:string']].dropna().drop_duplicates()
journals.rename(columns={'journal:string': 'name:ID'}, inplace=True)
journals.to_csv('processed_csv/nodes_journal.csv', index=False, sep=';')
print(f"Extracted {len(journals)} unique Journals.")

# --- 2. EXTRACT VOLUMES ---
# Select the columns and drop completely empty rows
volumes = df[['journal:string', 'volume:string', 'year:int']].dropna(subset=['journal:string', 'volume:string']).drop_duplicates()

# Ensure they are strings and handle potential NaNs before concatenating
volumes['journal_str'] = volumes['journal:string'].astype(str).fillna('')
volumes['volume_str'] = volumes['volume:string'].astype(str).fillna('')

# Create the unique ID
volumes['volume_id:ID'] = volumes['journal_str'] + "_Vol" + volumes['volume_str']
volumes.rename(columns={'volume:string': 'number:int', 'year:int': 'year:int'}, inplace=True)

# Keep only the columns for the Volume node
volumes_nodes = volumes[['volume_id:ID', 'number:int', 'year:int']]
volumes_nodes.to_csv('processed_csv/nodes_volume.csv', index=False, sep=';')
print(f"Extracted {len(volumes_nodes)} unique Volumes.")

# --- 3. CREATE RELATIONSHIPS ---
# Paper -> PUBLISHED_IN -> Volume
published_in = df[['article:ID', 'journal:string', 'volume:string']].dropna(subset=['journal:string', 'volume:string'])

# Repeat the ID creation logic here for consistency
published_in['journal_str'] = published_in['journal:string'].astype(str).fillna('')
published_in['volume_str'] = published_in['volume:string'].astype(str).fillna('')
published_in['volume_id:ID'] = published_in['journal_str'] + "_Vol" + published_in['volume_str']

published_in_rels = published_in[['article:ID', 'volume_id:ID']]
published_in_rels.rename(columns={'article:ID': ':START_ID', 'volume_id:ID': ':END_ID'}, inplace=True)
published_in_rels[':TYPE'] = 'PUBLISHED_IN'
published_in_rels.to_csv('processed_csv/rels_published_in.csv', index=False, sep=';')

# Volume -> BELONGS_TO -> Journal
belongs_to = volumes[['volume_id:ID', 'journal:string']]
belongs_to.rename(columns={'volume_id:ID': ':START_ID', 'journal:string': ':END_ID'}, inplace=True)
belongs_to[':TYPE'] = 'BELONGS_TO'
belongs_to.to_csv('processed_csv/rels_volume_belongs_to.csv', index=False, sep=';')

print("Extraction complete!")

Extracted 10 unique Journals.
Extracted 321 unique Volumes.
Extraction complete!


C:\Users\Uri\AppData\Local\Temp\ipykernel_15056\3576846929.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  published_in_rels.rename(columns={'article:ID': ':START_ID', 'volume_id:ID': ':END_ID'}, inplace=True)
C:\Users\Uri\AppData\Local\Temp\ipykernel_15056\3576846929.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  published_in_rels[':TYPE'] = 'PUBLISHED_IN'
C:\Users\Uri\AppData\Local\Temp\ipykernel_15056\3576846929.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: ht

## Proceedings from Conferences or Workshops

First of all, let us generate the clean inproceedings file

In [3]:
header_file = 'output_dblp/output_inproceedings_header.csv'
data_file = 'output_dblp/output_inproceedings.csv'

# 1. Get headers
with open(header_file, 'r', encoding='utf-8') as f:
    headers = f.read().strip().split(';') # Note: Your output showed ';' as the separator

# 2. Read a chunk of the data (e.g., first 100,000 rows to find good data)
df = pd.read_csv(data_file, sep=';', names=headers, nrows=100000, on_bad_lines='skip', low_memory=False)

# 3. Filter out the "junk" records
# Keep rows where the 'key' column does NOT contain 'dblpnote'
df_clean = df[~df['key:string'].astype(str).str.contains('dblpnote', na=False)]

# Keep rows that actually have a title and a year
df_clean = df_clean.dropna(subset=['title:string', 'year:int'])

# Print the total number of clean articles before subsetting
print(f"Total number of clean inproceedings: {len(df_clean)}")

# 4. Take a final sample of 10,000 clean papers for your lab
df_final = df_clean.head(10000)

# 5. Save the clean data to a new CSV
# df_final.to_csv('clean_inproceedings.csv', index=False, sep=';')

print(f"Successfully saved {len(df_final)} clean inproceedings!")

Total number of clean inproceedings: 99962
Successfully saved 10000 clean inproceedings!


In [ ]:
import pandas as pd

# Load the inproceedings (Assume you've already created a 'clean_inproceedings.csv')
df_conf = pd.read_csv('clean_inproceedings.csv', sep=';')

# --- 1. EXTRACT CONFERENCES ---
# booktitle:string represents the conference name in DBLP
conferences = df_conf[['booktitle:string']].dropna().drop_duplicates()
conferences.rename(columns={'booktitle:string': 'name:ID'}, inplace=True)
conferences['type:string'] = 'Conference' # You can manually refine this to 'Workshop' later
conferences.to_csv('processed_csv/nodes_conference.csv', index=False, sep=';')

# --- 2. EXTRACT EDITIONS ---
# An Edition is the combination of the Conference Name and the Year [cite: 37]
editions = df_conf[['booktitle:string', 'year:int']].dropna().drop_duplicates()
editions['edition_id:ID'] = editions['booktitle:string'].astype(str) + "_" + editions['year:int'].astype(str)
editions.rename(columns={'year:int': 'year:int'}, inplace=True)

# Note: DBLP doesn't usually provide City/Venue, so we will add a placeholder for now
editions['city:string'] = 'Unknown Venue' 

editions_nodes = editions[['edition_id:ID', 'year:int', 'city:string']]
editions_nodes.to_csv('processed_csv/nodes_edition.csv', index=False, sep=';')

# --- 3. CREATE RELATIONSHIPS ---
# Paper -> PRESENTED_AT -> Edition
presented_at = df_conf[['inproceedings:ID', 'booktitle:string', 'year:int']].dropna()
presented_at['edition_id:ID'] = presented_at['booktitle:string'].astype(str) + "_" + presented_at['year:int'].astype(str)
presented_at_rels = presented_at[['inproceedings:ID', 'edition_id:ID']]
presented_at_rels.rename(columns={'inproceedings:ID': ':START_ID', 'edition_id:ID': ':END_ID'}, inplace=True)
presented_at_rels[':TYPE'] = 'PRESENTED_AT'
presented_at_rels.to_csv('processed_csv/rels_presented_at.csv', index=False, sep=';')

# Edition -> BELONGS_TO -> Conference
belongs_to_conf = editions[['edition_id:ID', 'booktitle:string']]
belongs_to_conf.rename(columns={'edition_id:ID': ':START_ID', 'booktitle:string': ':END_ID'}, inplace=True)
belongs_to_conf[':TYPE'] = 'BELONGS_TO'
belongs_to_conf.to_csv('processed_csv/rels_edition_belongs_to.csv', index=False, sep=';')

C:\Users\Uri\AppData\Local\Temp\ipykernel_15056\3596605944.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  presented_at_rels.rename(columns={'inproceedings:ID': ':START_ID', 'edition_id:ID': ':END_ID'}, inplace=True)
C:\Users\Uri\AppData\Local\Temp\ipykernel_15056\3596605944.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  presented_at_rels[':TYPE'] = 'PRESENTED_AT'
C:\Users\Uri\AppData\Local\Temp\ipykernel_15056\3596605944.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentat

## Relations

Now that we have extracted from {output_articles.csv} and {output_inproceedings.csv} the information for nodes (Conference, Edition, Journal, Volume) and relations (Edition belongs to Conference, Volume belongs to Journal, Article presented at Edition, Article published at Volume); we must now cover the rest of relations. 

_WRITTEN BY_

In [8]:
import pandas as pd

df_final = pd.read_csv('processed_csv/clean_articles.csv', sep=';')

# Assuming df_final is your clean DataFrame of papers
# 1. Explode the author column
authors_exploded = df_final.assign(author=df_final['author:string[]'].str.split('|')).explode('author')
authors_exploded = authors_exploded.reset_index(drop=True)

# 2. Assign Corresponding Author (Boolean)
# We group by paper ID and mark the first author found as the corresponding one
authors_exploded['corresponding_author:boolean'] = False
authors_exploded.loc[authors_exploded.groupby('article:ID').head(1).index, 'corresponding_author:boolean'] = True

# 3. Save Relationship CSV
rels_wrote = authors_exploded[['author', 'article:ID', 'corresponding_author:boolean']]
rels_wrote.columns = [':START_ID', ':END_ID', 'corresponding_author:boolean']
rels_wrote[':TYPE'] = 'WRITTEN_BY'
rels_wrote.to_csv('processed_csv/rels_written_by.csv', index=False, sep=';')

false_count = (rels_wrote['corresponding_author:boolean'] == False).sum()
true_count = rels_wrote['corresponding_author:boolean'].sum()

print(f"From {len(rels_wrote)} 'has_written' relations, there are {true_count} corresponding authors and {false_count} non-corresponding")

From 28060 'has_written' relations, there are 10000 corresponding authors and 18060 non-corresponding


C:\Users\Uri\AppData\Local\Temp\ipykernel_11676\911882780.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rels_wrote[':TYPE'] = 'WRITTEN_BY'


In [9]:
import pandas as pd

# Load clean inproceedings
df_inproc = pd.read_csv('processed_csv/clean_inproceedings.csv', sep=';')

# Explode authors
inproc_authors = df_inproc.assign(author=df_inproc['author:string[]'].str.split('|')).explode('author')
inproc_authors = inproc_authors.reset_index(drop=True)

# Assign one Corresponding Author per paper
inproc_authors['corresponding_author:boolean'] = False
inproc_authors.loc[inproc_authors.groupby('inproceedings:ID').head(1).index, 'corresponding_author:boolean'] = True

# Save to the relationship file (Append if you already have articles, or save new)
rels_wrote_inproc = inproc_authors[['author', 'inproceedings:ID', 'corresponding_author:boolean']]
rels_wrote_inproc.columns = [':START_ID', ':END_ID', 'corresponding_author:boolean']
rels_wrote_inproc[':TYPE'] = 'WRITTEN_BY'
rels_wrote_inproc.to_csv('processed_csv/rels_wrote_inproceedings.csv', index=False, sep=';')

false_count = (rels_wrote_inproc['corresponding_author:boolean'] == False).sum()
true_count = rels_wrote_inproc['corresponding_author:boolean'].sum()

print(f"From {len(rels_wrote)} 'has_written' relations, there are {true_count} corresponding authors and {false_count} non-corresponding")

From 28060 'has_written' relations, there are 10000 corresponding authors and 26395 non-corresponding


C:\Users\Uri\AppData\Local\Temp\ipykernel_11676\3087125380.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rels_wrote_inproc[':TYPE'] = 'WRITTEN_BY'


_CITED BY_

In [ ]:
import pandas as pd

# 1. Load the raw citations with the correct columns
raw_cites = pd.read_csv('output_dblp/output_cite.csv', sep=';')

# Based on your output:
from_col = ':ID' 
to_col = 'cite:string'

# 2. Ensure we are comparing IDs as strings
# We use 'key:string' from df_final because 'cite:string' matches the DBLP keys (e.g., 'conf/stoc/...')
clean_paper_keys = set(df_final['key:string'].astype(str))

# 3. Filter: Keep only rows where BOTH the citing and cited paper exist in our 10,000 sample
rels_cites = raw_cites[
    raw_cites[from_col].astype(str).isin(clean_paper_keys) & 
    raw_cites[to_col].astype(str).isin(clean_paper_keys)
].copy()

# 4. Format for Neo4j Import
# We need to map these to the actual internal IDs (:ID) used in your node files
# Create a mapping dictionary from key -> article:ID
id_map = dict(zip(df_final['key:string'].astype(str), df_final['article:ID'].astype(str)))

rels_cites[':START_ID'] = rels_cites[from_col].map(id_map)
rels_cites[':END_ID'] = rels_cites[to_col].map(id_map)
rels_cites[':TYPE'] = 'CITED_BY'

# Drop the old columns and any rows that failed to map
rels_final = rels_cites[[':START_ID', ':END_ID', ':TYPE']].dropna()

# 5. Save
rels_final.to_csv('processed_csv/rels_cited_by.csv', index=False, sep=';')

print(f"Filtered to {len(rels_final)} internal citations within your sample.")

Filtered to 0 internal citations within your sample.


This raises a problem, since we previously subsampled 10000 rows and this has led us to not having any article citing another one. Thus, we will instead generate random citations for the sake of simplicity. 

In [ ]:
import random

# Load your clean papers
df_final = pd.read_csv('clean_articles.csv', sep=';')
paper_ids = df_final['article:ID'].tolist()

synthetic_cites = []

# For every paper, let's randomly cite 1 to 5 other papers from our 10,000 sample
for p_id in paper_ids:
    # Pick random papers to cite (excluding itself)
    potential_targets = [p for p in paper_ids if p != p_id]
    num_cites = random.randint(1, 5)
    
    cited_papers = random.sample(potential_targets, num_cites)
    
    for cited in cited_papers:
        synthetic_cites.append({
            ':START_ID': p_id,
            ':END_ID': cited,
            ':TYPE': 'CITED_BY'
        })

# Save to your relationship file
rels_final = pd.DataFrame(synthetic_cites)
rels_final.to_csv('processed_csv/rels_cited_by.csv', index=False, sep=';')

print(f"Generated {len(rels_final)} synthetic citations within your sample!")

Generated 29940 synthetic citations within your sample!


_COVERS_

Keeping in mind we have no actual information about keywords, what we will do is randomly assign topics to each paper from a pre-defined list.

In [ ]:
import pandas as pd
import random

# 1. Define our domain-specific keywords (MDS/SDM related)
db_keywords = [
    "data management", "indexing", "data modeling", "big data", "data processing", "data storage", "data querying",
    "query optimization", "nosql", "data mining", "computer vision", "machine learning", "distributed systems"
]

# 2. Save the Keyword Nodes
# Neo4j needs an ID (the word itself works well here)
nodes_keyword = pd.DataFrame({'word:ID': db_keywords})
nodes_keyword[':LABEL'] = 'Keyword'
nodes_keyword.to_csv('processed_csv/nodes_keyword.csv', index=False, sep=';')

# 3. Create the 'COVERS' relationships
# Load your paper IDs
df_final = pd.read_csv('processed_csv/clean_articles.csv', sep=';')
paper_ids = df_final['article:ID'].tolist()

keyword_rels = []

for p_id in paper_ids:
    # Randomly pick 1 to 3 keywords for this paper
    assigned = random.sample(db_keywords, random.randint(2, 5))
    
    for word in assigned:
        keyword_rels.append({
            ':START_ID': p_id, 
            ':END_ID': word,
            ':TYPE': 'COVERS'
        })

# 4. Save to CSV
rels_covers = pd.DataFrame(keyword_rels)
rels_covers.to_csv('processed_csv/rels_covers.csv', index=False, sep=';')

print(f"Created {len(db_keywords)} keyword nodes and {len(rels_covers)} COVERS relationships.")

Created 11 keyword nodes and 20046 COVERS relationships.


In [ ]:
import random

# Use the same list from before
db_keywords = [
    "data management", "indexing", "data modeling", "big data", "data processing", "data storage", "data querying",
    "query optimization", "nosql", "data mining", "computer vision", "machine learning", "distributed systems"
]
df_inproc = pd.read_csv('processed_csv/clean_inproceedings.csv', sep=';')
inproc_ids = df_inproc['inproceedings:ID'].tolist()

inproc_keyword_rels = []
for p_id in inproc_ids:
    assigned = random.sample(db_keywords, random.randint(1, 3))
    for word in assigned:
        inproc_keyword_rels.append({':START_ID': p_id, ':END_ID': word, ':TYPE': 'COVERS'})

pd.DataFrame(inproc_keyword_rels).to_csv('processed_csv/rels_covers_inproceedings.csv', index=False, sep=';')

_HAS REVIEWED_

Synthetically generated relations among authors and articles

In [ ]:
import pandas as pd
import random

# 1. Load your paper IDs and Author-Paper relationships
df_papers = pd.read_csv('clean_articles.csv', sep=';')
df_authors_wrote = pd.read_csv('processed_csv/rels_written_by.csv', sep=';')

# 2. Get unique lists
all_authors = df_authors_wrote[':START_ID'].unique().tolist()
paper_ids = df_papers['article:ID'].tolist()

review_rels = []

for p_id in paper_ids:
    # Get the authors of this specific paper to exclude them from reviewing it
    authors_of_this_paper = set(df_authors_wrote[df_authors_wrote[':END_ID'] == p_id][':START_ID'])
    
    # Potential reviewers are authors who didn't write this paper
    potential_reviewers = [a for a in all_authors if a not in authors_of_this_paper]
    
    # Randomly pick 3 reviewers per paper
    if len(potential_reviewers) >= 3:
        selected = random.sample(potential_reviewers, 3)
        for reviewer in selected:
            review_rels.append({
                ':START_ID': reviewer,
                ':END_ID': p_id,
                ':TYPE': 'HAS_REVIEWED'
            })

# 3. Save to CSV
pd.DataFrame(review_rels).to_csv('processed_csv/rels_has_reviewed.csv', index=False, sep=';')

print(f"Generated {len(review_rels)} simple HAS_REVIEWED relationships.")

Done. Generated 30000 simple HAS_REVIEWED relationships.


In [ ]:
all_authors = inproc_authors['author'].unique().tolist()
inproc_review_rels = []

for p_id in inproc_ids:
    # Exclude authors of the paper
    paper_authors = set(inproc_authors[inproc_authors['inproceedings:ID'] == p_id]['author'])
    potential = [a for a in all_authors if a not in paper_authors]
    
    if len(potential) >= 3:
        for reviewer in random.sample(potential, 3):
            inproc_review_rels.append({':START_ID': reviewer, ':END_ID': p_id, ':TYPE': 'HAS_REVIEWED'})

pd.DataFrame(inproc_review_rels).to_csv('processed_csv/rels_reviews_inproceedings.csv', index=False, sep=';')

_CITED BY_

Synthetically generate citations between both inproceedings and articles

In [4]:
import pandas as pd
import random

# 1. Load both sets of paper IDs
df_art = pd.read_csv('processed_csv/clean_articles.csv', sep=';')
df_inp = pd.read_csv('processed_csv/clean_inproceedings.csv', sep=';')

# 2. Combine all IDs into one master list
# article:ID and inproceedings:ID are just names; the values are what matter
all_paper_ids = df_art['article:ID'].tolist() + df_inp['inproceedings:ID'].tolist()

synthetic_cites = []

# 3. For every paper in the ENTIRE 20,000 sample
for p_id in all_paper_ids:
    # Potential targets are ANY other paper in the 20k sample (Journal or Conference)
    potential_targets = [p for p in all_paper_ids if p != p_id]
    
    # Let's give each paper 1 to 5 random citations
    num_cites = random.randint(1, 40)
    cited_papers = random.sample(potential_targets, num_cites)
    
    for cited in cited_papers:
        synthetic_cites.append({
            ':START_ID': p_id,
            ':END_ID': cited,
            ':TYPE': 'CITED_BY'
        })

# 4. Save to one single relationship file
rels_final = pd.DataFrame(synthetic_cites)
rels_final.to_csv('processed_csv/rels_cited_by_TOTAL.csv', index=False, sep=';')

print(f"Generated {len(rels_final)} universal citations linking Articles and Inproceedings!")

Generated 409550 universal citations linking Articles and Inproceedings!


Since articles and proceedings come from different dblp output files, we should unify all relations under the same node "Paper

In [10]:
# Load both keyword files
kw_art = pd.read_csv('processed_csv/rels_covers.csv', sep=';')
kw_inp = pd.read_csv('processed_csv/rels_covers_inproceedings.csv', sep=';')

# Merge and save
pd.concat([kw_art, kw_inp]).to_csv('processed_csv/rels_covers_TOTAL.csv', index=False, sep=';')

# Load both review files
rev_art = pd.read_csv('processed_csv/rels_has_reviewed.csv', sep=';')
rev_inp = pd.read_csv('processed_csv/rels_reviews_inproceedings.csv', sep=';')

# Merge and save
pd.concat([rev_art, rev_inp]).to_csv('processed_csv/rels_has_reviewed_TOTAL.csv', index=False, sep=';')

# Load both authorship files
wrote_art = pd.read_csv('processed_csv/rels_written_by.csv', sep=';')
wrote_inp = pd.read_csv('processed_csv/rels_wrote_inproceedings.csv', sep=';')

# Standardize columns (ensure they both have :START_ID, :END_ID, :TYPE, and the boolean)
wrote_inp.columns = wrote_art.columns

# Merge and save
pd.concat([wrote_art, wrote_inp]).to_csv('processed_csv/rels_written_by_TOTAL.csv', index=False, sep=';')

## Author nodes

In [ ]:
import pandas as pd
# Grab all unique authors from both relationship files
a1 = pd.read_csv('processed_csv/rels_written_by.csv', sep=';')[':START_ID']
a2 = pd.read_csv('processed_csv/rels_wrote_inproceedings.csv', sep=';')[':START_ID']
authors = pd.concat([a1, a2]).unique()
pd.DataFrame({'name:ID': authors, ':LABEL': 'Author'}).to_csv('processed_csv/nodes_author.csv', index=False, sep=';')

## Paper nodes

In [3]:
import pandas as pd

# Update Articles
df_art = pd.read_csv('processed_csv/clean_articles.csv', sep=';')
df_art['abstract:string'] = "This paper explores the concepts behind " + df_art['title:string'] + ". We present novel findings and methodologies."
df_art.to_csv('processed_csv/clean_articles.csv', index=False, sep=';')

# Update Inproceedings
df_inp = pd.read_csv('processed_csv/clean_inproceedings.csv', sep=';')
df_inp['abstract:string'] = "This study discusses " + df_inp['title:string'] + ", offering new perspectives for the conference attendees."
df_inp.to_csv('processed_csv/clean_inproceedings.csv', index=False, sep=';')